In [70]:
import ast
import csv
import json
import random
from pathlib import Path
import zipfile
from collections import OrderedDict
from pathlib import Path
from typing import Any

import torch



In [ ]:

PROJECT_ROOT = Path("C:/Users/ragha/OneDrive/Desktop/SDP-Sequential-Diagnosis/AFA_bench")

DDX_zip = PROJECT_ROOT.parent / "data" / "release_train_patients.zip"
evidence_json = PROJECT_ROOT.parent / "data" / "release_evidences.json"

output_root = Path("C:/Users/ragha/OneDrive/Desktop/SDP-Sequential-Diagnosis/AFA_bench/extra/data/misc")
pain_feature = "pain"

N_patients = 20_000
selection_seed = 42
split_seed = 42

train_frac = 0.80
val_frac = 0.10
test_frac = 0.10

E53 = "E_53"
E55 = "E_55"

EXCLUDED_FIELDS = {
    "PATHOLOGY",
    "DIFFERENTIAL_DIAGNOSIS",
    "EVIDENCES",
    "INITIAL_EVIDENCE",
}

### utilities


In [72]:
def parse_literal(value: str) -> Any:
    value = value.strip()
    if not value:
        return None
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError) as exc:
        raise ValueError(f"Could not parse DDXPlus value: {value[:200]}") from exc

def load_raw_patients() -> list[dict[str, str]]:
    if not DDX_zip.exists():
        raise FileNotFoundError(f"Missing DDXPlus ZIP: {DDX_ZIP}")
    if not evidence_json.exists():
        raise FileNotFoundError(f"Missing evidence definitions: {EVIDENCE_JSON}")

    with zipfile.ZipFile(DDX_zip) as zf:
        names = [name for name in zf.namelist() if not name.endswith("/")]
        if len(names) != 1:
            raise ValueError(
                f"Expected exactly one file in the ZIP, found: {names}"
            )

        with zf.open(names[0]) as raw_file:
            text = raw_file.read().decode("utf-8")

    rows = list(csv.DictReader(text.splitlines()))
    required = {
        "AGE",
        "SEX",
        "PATHOLOGY",
        "EVIDENCES",
        "INITIAL_EVIDENCE",
        "DIFFERENTIAL_DIAGNOSIS",
    }
    missing = required - set(rows[0].keys()) if rows else required
    if missing:
        raise ValueError(f"Missing raw DDXPlus columns: {sorted(missing)}")
    return rows

def load_evidence_definitions() -> dict[str, dict[str, Any]]:
    with evidence_json.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise TypeError("release_evidences.json must contain an object/dict")
    return data

def select_patients(rows):
    rng = random.Random(selection_seed)

    indices = list(range(len(rows)))
    rng.shuffle(indices)

    return [rows[i] for i in indices[:N_patients]]


### feature cleaning


In [73]:
def evidence_question_value_key(evidence_id, value_id):
    if value_id is None:
        return evidence_id
    return f"{evidence_id}@{value_id}"


def build_feature_vocabulary(selected_rows, evidence_defs):
    features = OrderedDict()

    features["age"] = None
    features["sex"] = None
    features["pain"] = None

    for row in selected_rows:
        evidence_list = parse_literal(row["EVIDENCES"]) or []

        for item in evidence_list:

            if "_@_" in item:
                evidence_id, value_id = item.split("_@_", 1)

                if evidence_id == "E_55":
                    continue

                if evidence_id in evidence_defs:
                    if evidence_defs[evidence_id].get("data_type") == "M":
                        features[f"{evidence_id}@{value_id}"] = None
                    else:
                        features[evidence_id] = None

            else:
                evidence_id = item

                if evidence_id in ("E_53", "E_55"):
                    continue

                if evidence_id in evidence_defs:
                    features[evidence_id] = None

    return ["age", "sex", "pain"] + sorted(
        f for f in features
        if f not in ("age", "sex", "pain")
    )

### feature representation

In [74]:
def extract_patient_evidence(row):
    value = parse_literal(row["EVIDENCES"])

    if not isinstance(value, list):
        raise TypeError("EVIDENCES must parse to a list.")

    return [x for x in value if isinstance(x, str)]


def pain_is_present(row):
    evidence = extract_patient_evidence(row)

    if any(
        item == E53 or item.startswith(E53 + "_@_")
        for item in evidence
    ):
        return True

    for item in evidence:
        if item.startswith(E55 + "_@_"):
            _, value_id = item.split("_@_", 1)

            if value_id != "V_123":
                return True

        elif item == E55:
            return True

    return False


def encode_sex(sex):
    mapping = {
        "F": 0.0,
        "M": 1.0,
    }

    sex = sex.strip().upper()

    if sex not in mapping:
        raise ValueError(f"Unexpected SEX value: {sex!r}")

    return mapping[sex]


def raw_value_feature_name(item):
    if "_@_" in item:
        evidence_id, value_id = item.split("_@_", 1)

        if evidence_id in (E53, E55):
            return None

        return evidence_question_value_key(evidence_id, value_id)

    if item in (E53, E55):
        return None

    return item


def encode_row(row, feature_names):
    values = {name: 0.0 for name in feature_names}

    values["age"] = float(row["AGE"])
    values["sex"] = encode_sex(row["SEX"])
    values["pain"] = 1.0 if pain_is_present(row) else 0.0

    for item in extract_patient_evidence(row):
        name = raw_value_feature_name(item)

        if name is not None and name in values:
            values[name] = 1.0

    return [values[name] for name in feature_names]

### save final dataset

In [75]:
import pandas as pd


def make_clean_dataframe(rows, feature_names, label_mapping):
    data = []

    for row in rows:
        features = encode_row(row, feature_names)
        label = label_mapping[row["PATHOLOGY"].strip()]
        data.append(features + [label])

    return pd.DataFrame(
        data,
        columns=feature_names + ["pathology"],
    )


def main():
    rows = load_raw_patients()
    evidence_defs = load_evidence_definitions()

    selected_rows = select_patients(rows)

    feature_names = build_feature_vocabulary(
        selected_rows,
        evidence_defs,
    )

    label_mapping = build_label_mapping(selected_rows)

    df = make_clean_dataframe(
        selected_rows,
        feature_names,
        label_mapping,
    )

    output_root.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        output_root / "ddxplus.csv",
        index=False,
    )

    save_json(
        label_mapping,
        output_root / "label_mapping.json",
    )

    print("Done")
    print("Patients:", len(df))
    print("Features:", len(feature_names))
    print("Classes:", len(label_mapping))
    print("Shape:", df.shape)


main()

Done
Patients: 20000
Features: 333
Classes: 49
Shape: (20000, 334)
